<a href="https://www.kaggle.com/code/amoghpanthangi/spml-2b?scriptVersionId=306157689" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# **FOOD IMAGE CLASSIFICATION USING CNN**

## **REQUIREMENTS**

In [1]:
import torch
import torch.nn.modules as nn
import numpy as np
import pandas
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torchvision.transforms import AutoAugment
from torch.utils.data import DataLoader, random_split, Subset
import h5py
from sklearn.model_selection import train_test_split
import pandas as pd


## **IMAGE PREPROCESSING AND DATASET CREATION**

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')   

trans = transforms.Compose([transforms.Resize((64, 64)), transforms.ToTensor()]) #resizing to save time in this calculation
calc_dset = datasets.ImageFolder(root='/kaggle/input/food41/images', transform = trans)
calc_dloader = DataLoader(calc_dset, batch_size=64, shuffle=False, num_workers=4)

sum_ = torch.zeros(3)
square_sum = torch.zeros(3)
total_pixels=0.0
for image, label in calc_dloader:
    pixel_count=image.size(0)*image.size(2)*image.size(3)             #calculating the mean and std for this dataset for accurate normalization
    sum_ += image.sum(dim=[0, 2, 3])
    square_sum += (image ** 2).sum(dim=[0, 2, 3])
    total_pixels += pixel_count
mean = sum_/total_pixels
std = (square_sum/total_pixels - mean**2).sqrt()

print(mean)
print(std)



tensor([0.5458, 0.4443, 0.3443])
tensor([0.2601, 0.2619, 0.2674])


In [3]:
test_trans = transforms.Compose([transforms.Resize((224, 224)),
                                  transforms.ToTensor(), 
                                  transforms.Normalize(mean, std)])
train_trans = transforms.Compose([transforms.Resize((224, 224)),
                                  transforms.RandomHorizontalFlip(0.5),
                                  transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),  #introducing augmentation to avoid generalising while training
                                  transforms.ToTensor(), 
                                  transforms.Normalize(mean, std)])

full_dset = datasets.ImageFolder(root='/kaggle/input/food41/images')

train_size = int(0.8*len(full_dset))
val_size = int(0.1*len(full_dset))     #splitting
test_size = len(full_dset) - train_size - val_size
train_id, val_id, test_id = random_split(range(len(full_dset)), [train_size, val_size, test_size])

train_dataset = datasets.ImageFolder(root='/kaggle/input/food41/images', transform=train_trans)
val_dataset   = datasets.ImageFolder(root='/kaggle/input/food41/images', transform=test_trans)
test_dataset  = datasets.ImageFolder(root='/kaggle/input/food41/images', transform=test_trans)

train_dset = Subset(train_dataset, train_id.indices)
val_dset   = Subset(val_dataset, val_id.indices)   #to make sure only training set has the augment
test_dset  = Subset(test_dataset, test_id.indices)

train_dloader = DataLoader(train_dset, batch_size=32, shuffle=True, num_workers=4)
val_dloader = DataLoader(val_dset, batch_size=32, shuffle=False, num_workers=4)
test_dloader = DataLoader(test_dset, batch_size=64, shuffle=False, num_workers=4)



## **CONVOLUTIONAL NEURAL NETWORK ARCHITECTURE**

In [4]:
class ResidualBlock(nn.Module):
    def __init__(self, in_, bottleneck, out_):
        super().__init__()
        #1x1 covolution to reduce dimension
        self.layer1 = nn.Sequential(             
            nn.Conv2d(in_, bottleneck, kernel_size=1),
            nn.BatchNorm2d(bottleneck),
            nn.ReLU()
        )
        #3x3 convolution to process features
        self.layer2 = nn.Sequential(
            nn.Conv2d(bottleneck, bottleneck, kernel_size=3, padding=1),
            nn.BatchNorm2d(bottleneck),
            nn.ReLU()
        )
        #restoring dimensions
        self.layer3 = nn.Sequential(                              #Residual Bottleneck block 
            nn.Conv2d(bottleneck, out_, kernel_size=1),
            nn.BatchNorm2d(out_)
        )
        self.relu = nn.ReLU()

        #A shortcut connection to match dimensions 
        if in_ != out_ :
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_, out_, kernel_size=1),
                nn.BatchNorm2d(out_)
            )
        else:
            self.shortcut = nn.Identity()        #if not needed preseve original

    def forward(self, x):
        identity = self.shortcut(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = x + identity
        x = self.relu(x)
        return x
    
        
class Model(nn.Module):
    def __init__(self):
        super().__init__()

        self.cblock1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),     #first convolution block
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.res_bottle1 = ResidualBlock(32, 32, 64)

        self.res_bottle2 = ResidualBlock(64, 64, 128)             #Residual bottlenack blocks

        self.res_bottle3 = ResidualBlock(128, 128, 256)

        self.cblock2 = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),                                 #second convolutional block
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.global_pool = nn.AdaptiveAvgPool2d((1, 1)) #Average pooling reducing dimensions to 1x1 to make process less expensive

        self.dblock1 = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256),                 #dense block of linear connected layers
            nn.ReLU(),
            nn.Dropout(p=0.3)
        )

        self.output_layer = nn.Linear(256, 101)

    def forward(self, x):
        x = self.cblock1(x)
        x = self.res_bottle1(x)
        x = self.res_bottle2(x)
        x = self.res_bottle3(x)
        x = self.cblock2(x)
        x = self.global_pool(x)
        x = self.dblock1(x)
        x = self.output_layer(x)
        return x

model = Model().to(device)      



## **TRAINING AND VALIDATION**

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.001)

epochs=25
epoch_count=[]
train_losses=[]
val_losses=[]
train_accs=[]
val_accs=[]

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    train_correct = 0.0
    train_total = 0.0

    for input, out_true in train_dloader:
        input, out_true = input.to(device), out_true.to(device)
        out_pred = model(input)
        prediction = out_pred.argmax(dim=1)
        train_correct += (prediction == out_true).sum().item()
        batch_avg_loss = loss_fn(out_pred, out_true)
        optimizer.zero_grad()
        batch_avg_loss.backward()
        optimizer.step()
        train_loss += batch_avg_loss.item()
        train_total += out_true.size(0)

    train_acc = (train_correct / train_total) * 100
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    model.eval() 
    val_loss = 0.0
    val_correct = 0.0      
    val_total = 0.0

    with torch.inference_mode():
        
        for val_input, val_true in val_dloader:
            val_input, val_true = val_input.to(device), val_true.to(device)
            val_pred = model(val_input)
            predict_val=val_pred.argmax(dim=1)
            val_correct += (predict_val==val_true).sum().item()
            val_avg_loss = loss_fn(val_pred,val_true)
            val_loss+=val_avg_loss.item()
            val_total+=val_true.size(dim=0)
        
    val_acc = (val_correct/val_total)*100
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    epoch_count.append(epoch+1)
    print(f"Training Loss={train_loss:.2f}, Training Acc={train_acc:.2f}%, Val Loss={val_loss:.2f}, Val acc={val_acc:.2f}%")


In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epoch_count, train_losses, label='Train Loss')
plt.plot(epoch_count, val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss Over Epochs')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epoch_count, train_accs, label='Train Acc')
plt.plot(epoch_count, val_accs, label='Val Acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Accuracy Over Epochs')
plt.legend()

plt.tight_layout()
plt.show()


## **MODEL SAVING AND OUTPUT GENERATION**

In [ ]:
import pickle
with open("model_weights.pkl", "wb") as f:
    pickle.dump(model.state_dict(), f)  #saving weights using pickle

In [ ]:
with open("/kaggle/input/food41/meta/meta/classes.txt", "r") as f:
    class_names = [line.strip() for line in f.readlines()]             #getting class names as in the txt each row has one class name

num_examples = 5
example_top5_labels = []

In [ ]:
model.eval()
with torch.inference_mode() :

    labels = []
    image_ids = []
    image_number = 0                      
    test_correct=0
    test_total=0
    shown = 0

    for images, answers in test_dloader:
        images, answers = images.to(device), answers.to(device)
        output = model(images)
        top5 = torch.topk(output, k=5, dim=1)
        top5_indices = top5.indices

        for i in range(images.size(0)):
            if shown >= num_examples:
                break
            class_indices = top5_indices[i].cpu().tolist()  #fetching indices
            class_names_top5 = [class_names[idx] for idx in class_indices]    #fetching class name using index  
            example_top5_labels.append(class_names_top5) #list of top fives
            shown += 1

        if shown >= num_examples:
            break
        
        predicted_labels = output.argmax(dim=1)
        for i in range(len(answers)):
            if predicted_labels[i] == answers[i]:
              test_correct+=1
            labels.append(predicted_labels[i].item())    
            image_ids.append(image_number) 
            image_number += 1
        test_total+=answers.size(dim=0)
    test_acc = (test_correct/test_total)*100
    print(f"Test Accuracy = {test_acc:.2f}%")

submission = pd.DataFrame({
    "id": image_ids,          
    "label": labels
})
submission.to_csv("submission.csv", index=False)

In [ ]:
for i, top5_names in enumerate(example_top5_labels):
    print(f"Example {i+1}:")
    for rank, name in enumerate(top5_names, start=1):
        print(f"  {rank}. {name}")